# 01 — anchor-op quickstart

Five-minute end-to-end tour on synthetic ground truth. Shows the essential pipeline:
draw a known Jacobian → simulate guide responses → measure → inspect the report.

**Prerequisites:** `PYTHONPATH=src` set, `numpy` + `matplotlib` installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import anchorop as ao

rng = np.random.default_rng(0)

## 1. Build a known ground truth

We construct a 6-dimensional Jacobian `J_true` with a mixed spectrum: two oscillatory 2×2 blocks,
one strongly damped mode, and one weakly hyperbolic mode. This gives a rich but interpretable target.

In [ ]:
def build_synthetic_J(d=6):
    J = np.zeros((d, d))
    J[0:2, 0:2] = np.array([[-0.4,  0.9], [-0.9, -0.4]])   # oscillatory
    J[2:4, 2:4] = np.array([[-0.6,  1.2], [-1.2, -0.6]])   # oscillatory
    J[4, 4] = -1.5                                          # damped
    J[5, 5] = -0.1                                          # weakly hyperbolic
    return J

J_true = build_synthetic_J(d=6)
eig_true = np.linalg.eigvals(J_true)
print("J_true eigenvalues:", eig_true)

## 2. Simulate guide responses

For each of 18 guides (3 per target gene), draw a knockdown efficiency κ ∈ [0.3, 0.9] and
compute the linear-response Δz = −J⁻¹ u, where u = −κ · e_target. Add small Gaussian noise.

In [ ]:
d, n_guides = 6, 18
targets = np.repeat(np.arange(d), 3)                    # 3 guides per target gene
kappas = 0.3 + 0.6 * rng.uniform(size=n_guides)
U = np.column_stack([-k * np.eye(d)[t] for k, t in zip(kappas, targets)])
S = -np.linalg.solve(J_true, U) + 0.02 * rng.normal(size=(d, n_guides))

names = [f"g{i}" for i in range(n_guides)]
effs = {n: float(kappas[i]) for i, n in enumerate(names)}
print(f"S shape: {S.shape},  U shape: {U.shape}")

## 3. Measure the operator

`measure_from_sensitivity` is the low-level entry point (no AnnData required). The `rank_tol=1e-2` guard
is preregistered — a singular direction of S must exceed 1% of σ_max to be treated as identified.

In [ ]:
m = ao.measure_from_sensitivity(
    S, U,
    guide_names=names,
    guide_efficiencies=effs,
    reg="tsvd", reg_param="path",
    rank_tol=1e-2,
)

r = m.report
print(f"effective response rank: {r.effective_response_rank}/{r.d}")
print(f"full domain identified:  {r.full_domain_identified}")
print(f"condition number:        {r.condition_number:.2f}")
print(f"retained guides:         {r.n_guides_retained}/{r.n_guides_input}")

## 4. Inspect the measured operator

Because `full_domain_identified` is `True`, we can access the full Jacobian `.J`. On a partial-rank
measurement this raises `IdentifiabilityError` — that's the type-level guard.

In [ ]:
J_fit = m.J   # would raise if full_domain_identified were False
rel_err = np.linalg.norm(J_fit - J_true) / np.linalg.norm(J_true)
print(f"||J_fit - J_true||_F / ||J_true||_F = {rel_err:.4f}")

eig_fit = np.linalg.eigvals(J_fit)
print(f"J_fit eigenvalues (first 3): {eig_fit[:3]}")

## 5. Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

vmax = max(abs(J_true).max(), abs(J_fit).max())
im = axes[0].imshow(np.hstack([J_true, np.full((d, 1), np.nan), J_fit]),
                     cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
axes[0].set_xticks([d/2, d + 1 + d/2]); axes[0].set_xticklabels(["J_true", "J_fit"])
axes[0].set_yticks([]); axes[0].set_title("Operator heatmaps")
fig.colorbar(im, ax=axes[0], shrink=0.8)

axes[1].scatter(eig_true.real, eig_true.imag, s=100, marker="o", facecolors="none",
                edgecolors="#1f4e79", lw=2, label="J_true")
axes[1].scatter(eig_fit.real, eig_fit.imag, s=60, marker="x", color="#c65a30", label="J_fit")
axes[1].axhline(0, color="0.6", lw=0.5); axes[1].axvline(0, color="0.6", lw=0.5)
axes[1].set_xlabel("Re(λ)"); axes[1].set_ylabel("Im(λ)")
axes[1].set_title("Eigenvalues"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.show()

## What just happened

- We drew a known 6-dim Jacobian, simulated 18 guide responses with 2% noise, and recovered `J_fit`
  to relative Frobenius error ≈ 5%.
- The report contains identifiability information: rank, condition, retained/dropped guides, path.
- Access to the full `.J` was safe because the report certified full-domain identification.

## Next steps

- **02**: how efficiency estimators work when your data comes from an actual Perturb-seq screen.
- **03**: the full pipeline from an AnnData object (not just precomputed S, U matrices).
- **04**: how to test whether the linear model is defensible on your data (and how NOT to test that).
- **05**: comparing anchor-op's operator against inferred operators from continuous-inference tools.